In [1]:
# IMPORTS

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report
from xgboost import XGBClassifier

In [2]:
# DATA LOADING

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [3]:
train.shape

(691369, 14)

In [4]:
test.shape

(296302, 13)

In [5]:
# CONFIGURATION

TRAIN_PATH = "train.csv"
TEST_PATH = "test.csv"

TARGET = "addicted_label"
ID_COL = "id"

N_SPLITS = 5
RANDOM_STATE = 42


In [6]:
# DATA PREPROCESSING

def preprocess_data(train_df, test_df):

    train_df = train_df.copy()
    test_df = test_df.copy()
    
    stress_map = {
        "Low": 0,
        "Medium": 1,
        "High": 2
    }

    impact_map = {
        "No": 0,
        "Yes": 1
    }

    for df in (train_df, test_df):
        if "stress_level" in df.columns:
            df["stress_level"] = df["stress_level"].replace(stress_map)

        if "academic_work_impact" in df.columns:
            df["academic_work_impact"] = df["academic_work_impact"].replace(impact_map)

    numerical_cols = [
        "age",
        "daily_screen_time_hours",
        "social_media_hours",
        "gaming_hours",
        "work_study_hours",
        "sleep_hours",
        "notifications_per_day",
        "app_opens_per_day",
        "weekend_screen_time",
        "stress_level",
        "academic_work_impact"
    ]

    for df in (train_df, test_df):
        for col in numerical_cols:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors="coerce")

    train_median = {}

    for col in numerical_cols:
        if col in train_df.columns:
            train_median[col] = train_df[col].median()

    train_filled = train_df.copy()
    test_filled = test_df.copy()

    for col in numerical_cols:
        if col in train_df.columns:
            # If a column were completely NaN, use 0 as a safe fallback.
            median_value = train_median[col]
            if pd.isna(median_value):
                median_value = 0.0

            train_filled[col] = train_df[col].fillna(median_value)
            test_filled[col] = test_df[col].fillna(median_value)

    eps = 1e-3

    def add_engineered_features(df, filled):
        
        df = df.copy()
        df["social_media_ratio"] = (filled["social_media_hours"] / (filled["daily_screen_time_hours"] + eps))
        df["gaming_ratio"] = (filled["gaming_hours"] / (filled["daily_screen_time_hours"] + eps))
        df["work_study_ratio"] = (filled["work_study_hours"] / (filled["daily_screen_time_hours"] + eps))
        df["leisure_vs_productive"] = ((filled["social_media_hours"] + filled["gaming_hours"]) /(filled["work_study_hours"] + eps))
        df["sleep_deficit"] = (8 - filled["sleep_hours"]).clip(lower=0)
        df["weekend_weekday_diff"] = (filled["weekend_screen_time"] - filled["daily_screen_time_hours"])
        df["notifications_per_hour"] = (filled["notifications_per_day"] / (filled["daily_screen_time_hours"] + eps))
        df["app_opens_per_hour"] = (filled["app_opens_per_day"] /(filled["daily_screen_time_hours"] + eps))
        df["screen_to_sleep_ratio"] = (filled["daily_screen_time_hours"] /(filled["sleep_hours"] + eps))
        df["age_bucket"] = pd.cut(filled["age"], bins=[17, 21, 25, 29, 35], labels=[0, 1, 2, 3]).astype(float)

        return df

    train_df = add_engineered_features(train_df, train_filled)
    test_df = add_engineered_features(test_df, test_filled)

    if "gender" in train_df.columns and "gender" in test_df.columns:
        
        combined_gender = pd.concat([train_df["gender"], test_df["gender"]], axis=0, ignore_index=True)

        gender_encoded = pd.get_dummies(combined_gender, prefix="gender", drop_first=True, dtype=int)

        train_gender = gender_encoded.iloc[:len(train_df)].reset_index(drop=True)
        test_gender = gender_encoded.iloc[len(train_df):].reset_index(drop=True)

        train_df = train_df.drop(columns=["gender"]).reset_index(drop=True)
        test_df = test_df.drop(columns=["gender"]).reset_index(drop=True)

        train_df = pd.concat([train_df, train_gender], axis=1)

        test_df = pd.concat([test_df, test_gender], axis=1)

    train_df = train_df.drop(columns=[ID_COL, TARGET], errors="ignore")
    test_df = test_df.drop(columns=[ID_COL], errors="ignore")

    test_df = test_df.reindex(columns=train_df.columns, fill_value=0)

    for col in train_df.columns:
        if train_df[col].dtype == "bool":
            train_df[col] = train_df[col].astype(int)

        if test_df[col].dtype == "bool":
            test_df[col] = test_df[col].astype(int)

    return train_df, test_df


X, X_test = preprocess_data(train, test)
y = train[TARGET].astype(int)

print("\n" + "=" * 70)
print("PREPROCESSING COMPLETE")
print("=" * 70)
print(f"Features: {X.shape[1]}")
print(f"Train X: {X.shape}")
print(f"Test X : {X_test.shape}")

if list(X.columns) != list(X_test.columns):
    raise ValueError("Train and test feature columns do not match.")

print("\nTarget distribution:")
print(y.value_counts())
print("\nTarget percentage:")
print(y.value_counts(normalize=True))



PREPROCESSING COMPLETE
Features: 23
Train X: (691369, 23)
Test X : (296302, 23)

Target distribution:
addicted_label
1    490474
0    200895
Name: count, dtype: int64

Target percentage:
addicted_label
1    0.709424
0    0.290576
Name: proportion, dtype: float64


In [7]:
# STRATIFIED K-FOLD

skf = StratifiedKFold(
    n_splits= N_SPLITS,
    shuffle=True,
    random_state = RANDOM_STATE
)

fold_auc = []
fold_accuracy = []
oof = np.zeros(len(X), dtype=float)
test_pred = np.zeros(len(X_test), dtype=float)

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), start=1):

    print(f"\n----- Fold {fold}/{N_SPLITS} -----")

    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]

    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    model = XGBClassifier(
        n_estimators=2000,
        learning_rate=0.1,
        max_depth=5,
        min_child_weight=3,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="auc",
        random_state= RANDOM_STATE,
        n_jobs=-1,
        tree_method="hist"
    )

    model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=False)

    valid_prob = model.predict_proba(X_valid)[:, 1]
    valid_pred = (valid_prob >= 0.5).astype(int)

    oof[valid_idx] = valid_prob

    test_pred += (model.predict_proba(X_test)[:, 1] / N_SPLITS)

    fold_auc_value = roc_auc_score(y_valid, valid_prob)
    fold_accuracy_value = accuracy_score(y_valid, valid_pred)

    fold_auc.append(fold_auc_value)
    fold_accuracy.append(fold_accuracy_value)

    print(f"Fold {fold} Accuracy : {fold_accuracy_value:.6f}")
    print(f"Fold {fold} ROC-AUC  : {fold_auc_value:.6f}")

mean_accuracy = np.mean(fold_accuracy)
mean_auc = np.mean(fold_auc)
std_auc = np.std(fold_auc)

overall_oof_auc = roc_auc_score(y,oof)
oof_pred = (oof >= 0.5).astype(int)

print("\n")
print("CROSS-VALIDATION RESULTS\n")

for i, (acc, auc) in enumerate(zip(fold_accuracy, fold_auc),start=1):
    
    print(f"Fold {i}: "f"Accuracy={acc:.6f} | "f"ROC-AUC={auc:.6f}")

print("\nMean Accuracy      :", f"{mean_accuracy:.6f}")
print("Mean ROC-AUC       :", f"{mean_auc:.6f}")
print("Std ROC-AUC        :", f"{std_auc:.6f}")
print("Overall OOF ROC-AUC:", f"{overall_oof_auc:.6f}")

print("\nOOF Classification Report:")
print(classification_report(y, oof_pred, digits=6))

print("OOF Confusion Matrix:")
print(confusion_matrix(y, oof_pred))

print("\n")
print("TEST PREDICTION CHECK\n")

print("Minimum probability:", test_pred.min())
print("Maximum probability:", test_pred.max())
print("Mean probability   :", test_pred.mean())

if not np.isfinite(test_pred).all():
    raise ValueError("Test predictions contain NaN or infinite values.")


----- Fold 1/5 -----
Fold 1 Accuracy : 0.902570
Fold 1 ROC-AUC  : 0.963722

----- Fold 2/5 -----
Fold 2 Accuracy : 0.903431
Fold 2 ROC-AUC  : 0.964313

----- Fold 3/5 -----
Fold 3 Accuracy : 0.903098
Fold 3 ROC-AUC  : 0.964429

----- Fold 4/5 -----
Fold 4 Accuracy : 0.904422
Fold 4 ROC-AUC  : 0.965153

----- Fold 5/5 -----
Fold 5 Accuracy : 0.902562
Fold 5 ROC-AUC  : 0.964069


CROSS-VALIDATION RESULTS

Fold 1: Accuracy=0.902570 | ROC-AUC=0.963722
Fold 2: Accuracy=0.903431 | ROC-AUC=0.964313
Fold 3: Accuracy=0.903098 | ROC-AUC=0.964429
Fold 4: Accuracy=0.904422 | ROC-AUC=0.965153
Fold 5: Accuracy=0.902562 | ROC-AUC=0.964069

Mean Accuracy      : 0.903217
Mean ROC-AUC       : 0.964338
Std ROC-AUC        : 0.000474
Overall OOF ROC-AUC: 0.964337

OOF Classification Report:
              precision    recall  f1-score   support

           0   0.842479  0.820299  0.831241    200895
           1   0.927181  0.937179  0.932153    490474

    accuracy                       0.903217    691369


In [8]:
submission = pd.DataFrame({ID_COL: test[ID_COL].values, TARGET: test_pred})

assert len(submission) == len(test)
assert list(submission.columns) == [ID_COL, TARGET]
assert np.isfinite( submission[TARGET]).all()

submission.to_csv("submission.csv",index=False)

In [9]:
submission.head()

,id,addicted_label
0,691369,0.999806
1,691370,0.914919
2,691371,0.937633
3,691372,0.990286
4,691373,0.998085
